# CS781 Capstone Phase 7: Prior Case Retrieval (PCR)
## Submission 06: Knowledge Distillation from Model 4 Teacher (153.9M) to Model 3 Student (109.5M) [MANDATORY]

* **Task**: Prior Case Retrieval (PCR) — Retrieve Top 7 legally & semantically relevant prior cases.
* **Target Metric**: **Micro-F1@7** (Secondary: Micro-Precision@7, Micro-Recall@7).
* **Compulsory Course Requirement**: **Knowledge Distillation (KD)** transfers semantic retrieval representations from a large teacher to a compact student.
* **Teacher Model**:
  * Fine-Tuned Model 4 Bi-Encoder (`sarweshbarnwal/CS781-Legal-Models` -> `Phase_7/sub01_model4_full_ft_hierarchical/`, fallback to `model4-capacity-boundary-154m`).
  * 14 layers, hidden dimension $d=896$, GQA 4:1, Tied Embeddings (~153.9M parameters).
  * Strictly **FROZEN** in evaluation mode on `cuda:0`.
* **Student Model**:
  * Initialized from Phase 1 foundation model `sarweshbarnwal/CS781-Legal-Models/model3-statutory-mha-110m`.
  * 12 layers, hidden dimension $d=768$, Full 12:12 Multi-Head Attention (MHA), Tied Embeddings (~109.5M parameters).
  * Fully **TRAINABLE** on `cuda:1`.
* **Distillation Bridge**:
  * Learnable linear projection $W_{\text{proj}}: 768 \to 896$ mapping student representations directly into teacher latent space.
* **Multi-Task Distillation Objective**:
  $$\mathcal{L}_{\text{total}} = 0.5 \cdot \mathcal{L}_{\text{InfoNCE}}(\text{student}, \text{gold}) + 0.4 \cdot T^2 \cdot \mathcal{D}_{\text{KL}}\big(\text{Softmax}(z_{\text{teacher}} / T) \;\big\|\; \text{Softmax}(z_{\text{student}} / T)\big) + 0.1 \cdot \big(1 - \cos(W_{\text{proj}}(u_{\text{student}}), u_{\text{teacher}})\big)$$
  with distillation temperature $T = 2.0$.
* **Dual-T4 Asymmetric Acceleration**:
  * Teacher model isolated on `cuda:0` (zero backward memory overhead).
  * Student model & distillation bridge allocated on `cuda:1` for maximum batch throughput.
* **Continuous Cloud Sync**: Push checkpoints every 200 steps to `sarweshbarnwal/CS781-Legal-Models/Phase_7/sub06_model4_to_model3_distill/` with automatic crash recovery.


In [ ]:
# 1. Environment Dependencies & Hugging Face Authentication
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import gc
import json
import time
import math
import glob
import random
import zipfile
import shutil
import threading
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Defensive PyTorch & Transformers compatibility guard for Python 3.13 / Kaggle
import sys
import types
import importlib
import torch

try:
    _u = importlib.import_module("torch._utils")
except Exception:
    _u = types.ModuleType("torch._utils")

if not hasattr(_u, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device): return -1 if device.type == "cpu" else (device.index or 0)
        if isinstance(device, int): return device
        return 0
    _u._get_device_index = _get_device_index

if not hasattr(_u, "try_import"):
    def try_import(name):
        try:
            return importlib.import_module(name)
        except Exception:
            return None
    _u.try_import = try_import

def _fallback_getattr(name):
    if name == "try_import":
        return _u.try_import
    if name == "_get_device_index":
        return _u._get_device_index
    return lambda *args, **kwargs: None

_u.__getattr__ = _fallback_getattr
setattr(torch, "_utils", _u)
sys.modules["torch._utils"] = _u

try:
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

try:
    import transformers.distributed.checkpoint as hf_dist_cp
    hf_dist_cp._check_distributed_checkpointing_available = lambda: False
except Exception:
    pass

from huggingface_hub import HfApi, login, snapshot_download, hf_hub_download
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)
    print("✓ Hugging Face authentication successful.")
else:
    print("⚠ Running without Hugging Face login; authenticated pushes will be disabled.")


In [ ]:
# 2. Execution Profile Selection & Global Configuration
# Toggle PROFILE between "SMOKE" (fast syntax & flow test) and "CORE" (full production training)
PROFILE = "CORE"  # Options: "SMOKE", "CORE"

PROFILES = {
    "SMOKE": {
        "max_train_queries": 20,       # 20 queries for quick smoke verification
        "max_candidates": 100,         # 100 candidate cases
        "max_dev_queries": 15,         # 15 dev queries
        "max_test_queries": 15,        # 15 test queries
        "train_epochs": 1,
        "batch_size": 4,
        "eval_steps": 20,
        "save_steps": 20,
        "warmup_steps": 10,
        "max_chunks": 4,               # 4 chunks x 512 tokens
        "chunk_size": 512,
        "chunk_stride": 256,
        "description": "Fast Functional Verification (Smoke Mode)"
    },
    "CORE": {
        "max_train_queries": None,     # Full dataset
        "max_candidates": None,        # Full candidate pool
        "max_dev_queries": None,       # Full dev validation
        "max_test_queries": None,      # Full test set
        "train_epochs": 4,
        "batch_size": 16,              # Efficient multi-task batch size
        "eval_steps": 200,
        "save_steps": 200,             # Continuous cloud sync every 200 steps
        "warmup_steps": 500,
        "max_chunks": 8,               # 8 chunks x 512 tokens
        "chunk_size": 512,
        "chunk_stride": 256,
        "description": "Full Production Multi-GPU Distillation Training (Core Mode)"
    }
}

CFG = PROFILES[PROFILE]
print(f"=== Active Profile: {PROFILE} ({CFG['description']}) ===")

# Global Constants & Paths
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MASTER_REPO = "sarweshbarnwal/CS781-Legal-Models"
TEACHER_HF_SUBFOLDER = "Phase_7/sub01_model4_full_ft_hierarchical"
TEACHER_FALLBACK_SUBFOLDER = "model4-capacity-boundary-154m"
STUDENT_BASE_SUBFOLDER = "model3-statutory-mha-110m"
EXPERIMENT_NAME = "sub06_model4_to_model3_distill"
HUB_SUBFOLDER = f"Phase_7/{EXPERIMENT_NAME}"

OUTPUT_DIR = "/kaggle/working/phase7_sub06" if os.path.exists("/kaggle") else "./phase7_sub06"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "loss_and_metrics_plot.png")
METRICS_JSON_PATH = os.path.join(OUTPUT_DIR, "val_metrics.json")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Local artifacts directory: {OUTPUT_DIR}")
print(f"HF Hub destination: {HF_MASTER_REPO}/{HUB_SUBFOLDER}")


In [ ]:
# 3. Dual-T4 Asymmetric Hardware Allocation (Teacher on cuda:0, Student on cuda:1)
print("=== Hardware Acceleration Verification ===")
gpu_count = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | GPU Count: {gpu_count}")

if torch.cuda.is_available():
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU [{i}]: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB")

    if gpu_count >= 2:
        teacher_device = torch.device("cuda:0")
        student_device = torch.device("cuda:1")
        print("✓ Dual-T4 Asymmetric Allocation ACTIVE:")
        print(f"  - Teacher Model (Frozen 154M): allocated on {teacher_device}")
        print(f"  - Student Model (Trainable 110M): allocated on {student_device}")
    else:
        teacher_device = torch.device("cuda:0")
        student_device = torch.device("cuda:0")
        print(f"ℹ Single GPU mode: both teacher and student allocated on {teacher_device}.")

    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cudnn.benchmark = True
    print("✓ PyTorch 2.0 SDPA memory-efficient attention activated.")
else:
    teacher_device = torch.device("cpu")
    student_device = torch.device("cpu")
    print("⚠ No GPU detected. Running on CPU fallback.")


In [ ]:
# 4. Dataset Ingestion & Preprocessing (Exploration-Lab/CS-781-Capstone)
import json

raw_data_dir = os.path.join(OUTPUT_DIR, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

print("Downloading Phase 7 PCR dataset from Hugging Face...")
snapshot_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    local_dir=raw_data_dir,
    allow_patterns=["Phase_7_PCR/*"],
    token=HF_TOKEN
)

pcr_base = os.path.join(raw_data_dir, "Phase_7_PCR")
if not os.path.exists(pcr_base):
    for root, dirs, files in os.walk(raw_data_dir):
        if "train.json" in files:
            pcr_base = root
            break

print(f"PCR Data Directory: {pcr_base}")

def load_json_file(filename: str) -> dict:
    candidates = [
        os.path.join(pcr_base, filename),
        os.path.join(pcr_base, "Train", filename),
        os.path.join(pcr_base, "Dev", filename),
        os.path.join(pcr_base, "Test", filename),
    ]
    for path in candidates:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    for root, _, files in os.walk(pcr_base):
        if filename in files:
            with open(os.path.join(root, filename), "r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError(f"Could not locate {filename} under {pcr_base}")

train_queries_raw = load_json_file("train.json")
train_candidates_raw = load_json_file("train_candidates.json")
dev_queries_raw = load_json_file("dev.json")
dev_candidates_raw = load_json_file("dev_candidates.json")
test_queries_raw = load_json_file("test.json")
test_candidates_raw = load_json_file("test_candidates.json")

print(f"Loaded Train Queries: {len(train_queries_raw):,} | Train Candidates: {len(train_candidates_raw):,}")
print(f"Loaded Dev Queries:   {len(dev_queries_raw):,} | Dev Candidates:   {len(dev_candidates_raw):,}")
print(f"Loaded Test Queries:  {len(test_queries_raw):,} | Test Candidates:  {len(test_candidates_raw):,}")

def flatten_case_text(case_obj: Any) -> str:
    if isinstance(case_obj, str):
        return case_obj.strip()
    elif isinstance(case_obj, list):
        return " ".join([str(x).strip() for x in case_obj if str(x).strip()])
    elif isinstance(case_obj, dict):
        if "text" in case_obj:
            return flatten_case_text(case_obj["text"])
        vals = [flatten_case_text(v) for v in case_obj.values() if isinstance(v, (str, list))]
        return " ".join(vals)
    return str(case_obj).strip()

def process_query_dict(q_dict: dict, max_count: Optional[int] = None) -> List[dict]:
    items = []
    for qid, qdata in list(q_dict.items())[:max_count]:
        if isinstance(qdata, dict):
            text = flatten_case_text(qdata.get("text", qdata.get("query_text", "")))
            gold_cids = qdata.get("relevant_candidates", qdata.get("gold_candidates", []))
            if isinstance(gold_cids, str):
                gold_cids = [gold_cids]
        else:
            text = flatten_case_text(qdata)
            gold_cids = []
        items.append({"qid": str(qid), "text": text, "gold_cids": [str(c) for c in gold_cids]})
    return items

def process_candidate_dict(c_dict: dict, max_count: Optional[int] = None) -> Dict[str, str]:
    processed = {}
    for cid, cdata in list(c_dict.items())[:max_count]:
        processed[str(cid)] = flatten_case_text(cdata)
    return processed

def process_splits_aligned(q_dict: dict, c_dict: dict, max_queries: Optional[int], max_cands: Optional[int]):
    queries = process_query_dict(q_dict, max_queries)
    if max_cands is None:
        candidates = {str(cid): flatten_case_text(cdata) for cid, cdata in c_dict.items()}
        return queries, candidates

    gold_cids = {cid for q in queries for cid in q["gold_cids"] if cid in c_dict or str(cid) in c_dict}
    selected_cids = [str(cid) for cid in c_dict.keys() if str(cid) in gold_cids]
    for cid in c_dict.keys():
        if len(selected_cids) >= max_cands:
            break
        if str(cid) not in gold_cids:
            selected_cids.append(str(cid))

    if len(selected_cids) < 7:
        for cid in c_dict.keys():
            if str(cid) not in selected_cids:
                selected_cids.append(str(cid))
            if len(selected_cids) >= min(7, len(c_dict)):
                break

    candidates = {cid: flatten_case_text(c_dict[cid]) for cid in selected_cids}
    return queries, candidates

train_queries, train_candidates = process_splits_aligned(train_queries_raw, train_candidates_raw, CFG["max_train_queries"], CFG["max_candidates"])
dev_queries, dev_candidates = process_splits_aligned(dev_queries_raw, dev_candidates_raw, CFG["max_dev_queries"], CFG["max_candidates"])
test_queries, test_candidates = process_splits_aligned(test_queries_raw, test_candidates_raw, CFG["max_test_queries"], CFG["max_candidates"])

print(f"Profile: {PROFILE} => Train Queries: {len(train_queries)}, Dev Queries: {len(dev_queries)}, Test Queries: {len(test_queries)}")
print(f"Test Candidates in evaluation pool: {len(test_candidates):,}")


In [ ]:
# Defensive PyTorch & Transformers compatibility guard for Python 3.13 / Kaggle
import sys
import types
import importlib
import torch

try:
    _u = importlib.import_module("torch._utils")
except Exception:
    _u = types.ModuleType("torch._utils")

if not hasattr(_u, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device): return -1 if device.type == "cpu" else (device.index or 0)
        if isinstance(device, int): return device
        return 0
    _u._get_device_index = _get_device_index

if not hasattr(_u, "try_import"):
    def try_import(name):
        try:
            return importlib.import_module(name)
        except Exception:
            return None
    _u.try_import = try_import

def _fallback_getattr(name):
    if name == "try_import":
        return _u.try_import
    if name == "_get_device_index":
        return _u._get_device_index
    return lambda *args, **kwargs: None

_u.__getattr__ = _fallback_getattr
setattr(torch, "_utils", _u)
sys.modules["torch._utils"] = _u

try:
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

try:
    import transformers.distributed.checkpoint as hf_dist_cp
    hf_dist_cp._check_distributed_checkpointing_available = lambda: False
except Exception:
    pass

# 5. Tokenizer Initialization
from transformers import PreTrainedTokenizerFast

local_tok_dir = os.path.join(OUTPUT_DIR, "tokenizer")
os.makedirs(local_tok_dir, exist_ok=True)

try:
    print(f"Downloading tokenizer from {HF_MASTER_REPO}...")
    snapshot_download(
        repo_id=HF_MASTER_REPO,
        allow_patterns=[f"{STUDENT_BASE_SUBFOLDER}/*token*"],
        local_dir=local_tok_dir,
        token=HF_TOKEN
    )
except Exception as e:
    print(f"Specific tokenizer pattern notice: {e}. Downloading entire student subfolder...")
    snapshot_download(
        repo_id=HF_MASTER_REPO,
        allow_patterns=[f"{STUDENT_BASE_SUBFOLDER}/*"],
        local_dir=local_tok_dir,
        token=HF_TOKEN
    )

tok_json_path = os.path.join(local_tok_dir, STUDENT_BASE_SUBFOLDER, "tokenizer", "tokenizer.json")
if not os.path.exists(tok_json_path):
    tok_json_path = os.path.join(local_tok_dir, STUDENT_BASE_SUBFOLDER, "tokenizer.json")
if not os.path.exists(tok_json_path):
    tok_json_path = os.path.join(local_tok_dir, "tokenizer.json")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=tok_json_path,
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>"
)
tokenizer.padding_side = "right"
print(f"✓ Tokenizer loaded successfully. Vocab Size: {len(tokenizer):,}")


In [ ]:
# 6. Teacher Model (154M), Student Model (110M), and Distillation Bridge
# Defensive PyTorch & Transformers compatibility guard for Python 3.13 / Kaggle
import sys
import types
import importlib
import torch

try:
    _u = importlib.import_module("torch._utils")
except Exception:
    _u = types.ModuleType("torch._utils")

if not hasattr(_u, "_get_device_index"):
    def _get_device_index(device, optional=False, allow_cpu=False):
        if isinstance(device, str): device = torch.device(device)
        if isinstance(device, torch.device): return -1 if device.type == "cpu" else (device.index or 0)
        if isinstance(device, int): return device
        return 0
    _u._get_device_index = _get_device_index

if not hasattr(_u, "try_import"):
    def try_import(name):
        try:
            return importlib.import_module(name)
        except Exception:
            return None
    _u.try_import = try_import

def _fallback_getattr(name):
    if name == "try_import":
        return _u.try_import
    if name == "_get_device_index":
        return _u._get_device_index
    return lambda *args, **kwargs: None

_u.__getattr__ = _fallback_getattr
setattr(torch, "_utils", _u)
sys.modules["torch._utils"] = _u

try:
    import transformers.utils.import_utils as hf_import_utils
    hf_import_utils.is_torchvision_available = lambda: False
except Exception:
    pass

try:
    import transformers.distributed.checkpoint as hf_dist_cp
    hf_dist_cp._check_distributed_checkpointing_available = lambda: False
except Exception:
    pass

from transformers import LlamaConfig, LlamaModel

class LegalBiEncoder(nn.Module):
    """
    Two-Tower Legal Bi-Encoder with Hierarchical Chunk Pooling:
    - Encodes up to max_chunks (e.g. 8 chunks of 512 tokens with 256 stride).
    - Masked Mean Pooling within each chunk.
    - Masked Mean Pooling across active chunks into a single L2-normalized representation.
    """
    def __init__(self, backbone: LlamaModel, hidden_size: int, max_chunks: int = 8):
        super().__init__()
        self.backbone = backbone
        self.hidden_size = hidden_size
        self.max_chunks = max_chunks

    def encode_chunks(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, chunk_mask: torch.Tensor) -> torch.Tensor:
        B, K, L = input_ids.shape
        flat_ids = input_ids.view(B * K, L)
        flat_mask = attention_mask.view(B * K, L)

        chunk_reprs_list = []
        max_sub_batch = 4
        for s_idx in range(0, flat_ids.size(0), max_sub_batch):
            e_idx = min(s_idx + max_sub_batch, flat_ids.size(0))
            sub_ids = flat_ids[s_idx:e_idx]
            sub_mask = flat_mask[s_idx:e_idx]

            sub_out = self.backbone(input_ids=sub_ids, attention_mask=sub_mask)
            sub_hidden = sub_out.last_hidden_state

            mask_expanded = sub_mask.unsqueeze(-1).expand_as(sub_hidden).float()
            sum_hidden = torch.sum(sub_hidden * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            sub_repr = sum_hidden / sum_mask
            chunk_reprs_list.append(sub_repr)

        chunk_reprs = torch.cat(chunk_reprs_list, dim=0).view(B, K, -1)  # [B, K, D]

        # 2. Masked Mean Pooling across active chunks
        cmask_expanded = chunk_mask.unsqueeze(-1).expand_as(chunk_reprs).float()
        doc_hidden = torch.sum(chunk_reprs * cmask_expanded, dim=1)
        doc_mask = torch.clamp(cmask_expanded.sum(dim=1), min=1e-9)
        doc_vector = doc_hidden / doc_mask  # [B, D]

        return F.normalize(doc_vector, p=2, dim=-1)

    def forward(self, q_ids, q_attn, q_cmask, d_ids=None, d_attn=None, d_cmask=None):
        u = self.encode_chunks(q_ids, q_attn, q_cmask)
        if d_ids is not None:
            v = self.encode_chunks(d_ids, d_attn, d_cmask)
            return u, v
        return u


class DistillationBridge(nn.Module):
    """
    Linear Projection Bridge W_proj (768 -> 896):
    Aligns student latent space directly into teacher latent space for Cosine Similarity alignment.
    """
    def __init__(self, student_dim: int = 768, teacher_dim: int = 896):
        super().__init__()
        self.proj = nn.Linear(student_dim, teacher_dim)
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, student_repr: torch.Tensor) -> torch.Tensor:
        return self.proj(student_repr)


# ============================================================================
# A. Load Teacher Model (Model 4 Bi-Encoder, 154M, d=896)
# ============================================================================
print("--- Loading Teacher Model (Model 4) ---")
teacher_dir = os.path.join(OUTPUT_DIR, "teacher_model")
os.makedirs(teacher_dir, exist_ok=True)

loaded_teacher = False
try:
    print(f"Checking for fine-tuned teacher weights at {HF_MASTER_REPO}/{TEACHER_HF_SUBFOLDER}...")
    snapshot_download(
        repo_id=HF_MASTER_REPO,
        allow_patterns=[f"{TEACHER_HF_SUBFOLDER}/*"],
        local_dir=teacher_dir,
        token=HF_TOKEN
    )
    t_path = os.path.join(teacher_dir, TEACHER_HF_SUBFOLDER, "final_model")
    if not os.path.exists(t_path):
        t_path = os.path.join(teacher_dir, TEACHER_HF_SUBFOLDER)
    teacher_backbone = LlamaModel.from_pretrained(t_path, torch_dtype=torch.float32)
    loaded_teacher = True
    print("✓ Loaded fine-tuned Model 4 Teacher from Sub-01.")
except Exception as e:
    print(f"Notice: Fine-tuned Sub-01 teacher not found ({e}). Loading pretrained Model 4 base weights...")

if not loaded_teacher:
    snapshot_download(
        repo_id=HF_MASTER_REPO,
        allow_patterns=[f"{TEACHER_FALLBACK_SUBFOLDER}/*"],
        local_dir=teacher_dir,
        token=HF_TOKEN
    )
    t_path = os.path.join(teacher_dir, TEACHER_FALLBACK_SUBFOLDER)
    teacher_backbone = LlamaModel.from_pretrained(t_path, torch_dtype=torch.float32)
    print("✓ Loaded pretrained Model 4 base as Teacher.")

teacher_model = LegalBiEncoder(backbone=teacher_backbone, hidden_size=896, max_chunks=CFG["max_chunks"])
teacher_model.to(teacher_device)
teacher_model.eval()

# Strictly freeze all teacher parameters
for p in teacher_model.parameters():
    p.requires_grad = False
print(f"✓ Teacher Model (154M) FROZEN on {teacher_device}.")


# ============================================================================
# B. Load Student Model (Model 3 Statutory MHA, 110M, d=768)
# ============================================================================
print("--- Loading Student Model (Model 3) ---")
student_dir = os.path.join(OUTPUT_DIR, "student_model")
os.makedirs(student_dir, exist_ok=True)

snapshot_download(
    repo_id=HF_MASTER_REPO,
    allow_patterns=[f"{STUDENT_BASE_SUBFOLDER}/*"],
    local_dir=student_dir,
    token=HF_TOKEN
)
s_path = os.path.join(student_dir, STUDENT_BASE_SUBFOLDER)
student_backbone = LlamaModel.from_pretrained(s_path, torch_dtype=torch.float32)
student_backbone.gradient_checkpointing_enable()
print("✓ Gradient checkpointing enabled on student backbone.")

student_model = LegalBiEncoder(backbone=student_backbone, hidden_size=768, max_chunks=CFG["max_chunks"])
student_model.to(student_device)
student_model.train()

# Ensure all student parameters are trainable
for p in student_model.parameters():
    p.requires_grad = True

# Distillation Bridge
bridge = DistillationBridge(student_dim=768, teacher_dim=896).to(student_device)

t_params = sum(p.numel() for p in teacher_model.parameters())
s_params = sum(p.numel() for p in student_model.parameters())
b_params = sum(p.numel() for p in bridge.parameters())
print(f"=== Model Parameter Accounting ===")
print(f"  Teacher Parameters (Frozen):    {t_params:,}")
print(f"  Student Parameters (Trainable): {s_params:,}")
print(f"  Bridge Parameters (Trainable):  {b_params:,}")
print(f"  Total Trainable in Student:     {s_params + b_params:,}")


In [ ]:
# 7. Multi-Task Distillation Loss & Data Pipeline

class DistillationLoss(nn.Module):
    """
    Multi-Task Distillation Loss:
      L_total = 0.5 * L_InfoNCE(student, gold)
              + 0.4 * T^2 * KL(Softmax(z_teacher/T) || Softmax(z_student/T))
              + 0.1 * (1 - CosineSimilarity(W_proj(u_student), u_teacher))
    with temperature T = 2.0.
    """
    def __init__(self, temperature: float = 2.0, tau: float = 0.05, alpha_task: float = 0.5, alpha_kl: float = 0.4, alpha_cos: float = 0.1):
        super().__init__()
        self.T = temperature
        self.tau = tau
        self.alpha_task = alpha_task
        self.alpha_kl = alpha_kl
        self.alpha_cos = alpha_cos
        self.kl_div = nn.KLDivLoss(reduction="batchmean")

    def forward(
        self,
        u_student: torch.Tensor,
        v_student: torch.Tensor,
        u_teacher: torch.Tensor,
        v_teacher: torch.Tensor,
        bridge: nn.Module
    ) -> Tuple[torch.Tensor, dict]:
        """
        u_student, v_student: [B, 768] (on student_device)
        u_teacher, v_teacher: [B, 896] (moved to student_device)
        """
        B = u_student.shape[0]
        targets = torch.arange(B, device=u_student.device)

        # 1. Task Loss: InfoNCE / Multiple Negatives Ranking Loss on Student
        z_student = torch.matmul(u_student, v_student.T) / self.tau  # [B, B]
        loss_infonce = F.cross_entropy(z_student, targets)

        # 2. Knowledge Distillation: KL Divergence over soft ranking distributions
        z_teacher = torch.matmul(u_teacher, v_teacher.T) / self.tau  # [B, B]
        
        # Soft probabilities with temperature T
        p_teacher = F.softmax(z_teacher / self.T, dim=-1)
        log_p_student = F.log_softmax(z_student / self.T, dim=-1)
        
        loss_kl = (self.T ** 2) * self.kl_div(log_p_student, p_teacher)

        # 3. Latent Geometry Alignment: Cosine Similarity between W_proj(u_student) and u_teacher
        u_student_proj = F.normalize(bridge(u_student), p=2, dim=-1)
        v_student_proj = F.normalize(bridge(v_student), p=2, dim=-1)
        
        u_teacher_norm = F.normalize(u_teacher, p=2, dim=-1)
        v_teacher_norm = F.normalize(v_teacher, p=2, dim=-1)

        cos_u = F.cosine_similarity(u_student_proj, u_teacher_norm, dim=-1).mean()
        cos_v = F.cosine_similarity(v_student_proj, v_teacher_norm, dim=-1).mean()
        loss_cos = 1.0 - 0.5 * (cos_u + cos_v)

        total_loss = (
            self.alpha_task * loss_infonce +
            self.alpha_kl * loss_kl +
            self.alpha_cos * loss_cos
        )

        metrics = {
            "loss_total": total_loss.item(),
            "loss_infonce": loss_infonce.item(),
            "loss_kl": loss_kl.item(),
            "loss_cos": loss_cos.item()
        }
        return total_loss, metrics


distill_criterion = DistillationLoss(temperature=2.0, tau=0.05)


def chunk_document(
    text: str,
    tokenizer,
    max_chunks: int = 8,
    chunk_size: int = 512,
    chunk_stride: int = 256
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    enc = tokenizer(
        text,
        truncation=False,
        add_special_tokens=True,
        return_attention_mask=True
    )
    all_input_ids = enc["input_ids"]
    all_attn_mask = enc["attention_mask"]
    total_len = len(all_input_ids)

    chunks_ids = []
    chunks_mask = []

    if total_len <= chunk_size:
        pad_len = chunk_size - total_len
        chunks_ids.append(all_input_ids + [tokenizer.pad_token_id] * pad_len)
        chunks_mask.append(all_attn_mask + [0] * pad_len)
    else:
        start = 0
        while start < total_len and len(chunks_ids) < max_chunks:
            end = min(start + chunk_size, total_len)
            c_ids = all_input_ids[start:end]
            c_mask = all_attn_mask[start:end]
            if len(c_ids) < chunk_size:
                pad_len = chunk_size - len(c_ids)
                c_ids = c_ids + [tokenizer.pad_token_id] * pad_len
                c_mask = c_mask + [0] * pad_len
            chunks_ids.append(c_ids)
            chunks_mask.append(c_mask)
            if end >= total_len:
                break
            start += chunk_stride

    num_actual = len(chunks_ids)
    while len(chunks_ids) < max_chunks:
        chunks_ids.append([tokenizer.pad_token_id] * chunk_size)
        chunks_mask.append([0] * chunk_size)

    chunk_mask = [1] * num_actual + [0] * (max_chunks - num_actual)

    return (
        torch.tensor(chunks_ids, dtype=torch.long),
        torch.tensor(chunks_mask, dtype=torch.long),
        torch.tensor(chunk_mask, dtype=torch.long)
    )


class LegalPCRDataset(Dataset):
    def __init__(self, queries: List[dict], candidates: Dict[str, str], tokenizer, cfg: dict):
        self.tokenizer = tokenizer
        self.cfg = cfg
        self.samples = []
        cand_keys = list(candidates.keys())

        for q in queries:
            qid = q["qid"]
            q_text = q["text"]
            gold_cids = q.get("gold_cids", [])
            valid_golds = [cid for cid in gold_cids if cid in candidates]

            if valid_golds:
                for g_cid in valid_golds:
                    self.samples.append({
                        "qid": qid,
                        "query_text": q_text,
                        "cand_id": g_cid,
                        "cand_text": candidates[g_cid]
                    })
            elif cand_keys:
                self.samples.append({
                    "qid": qid,
                    "query_text": q_text,
                    "cand_id": cand_keys[0],
                    "cand_text": candidates[cand_keys[0]]
                })

        if len(self.samples) < cfg["batch_size"] and self.samples:
            while len(self.samples) < cfg["batch_size"]:
                self.samples.append(self.samples[0])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        q_ids, q_attn, q_cmask = chunk_document(
            s["query_text"], self.tokenizer,
            max_chunks=self.cfg["max_chunks"],
            chunk_size=self.cfg["chunk_size"],
            chunk_stride=self.cfg["chunk_stride"]
        )
        d_ids, d_attn, d_cmask = chunk_document(
            s["cand_text"], self.tokenizer,
            max_chunks=self.cfg["max_chunks"],
            chunk_size=self.cfg["chunk_size"],
            chunk_stride=self.cfg["chunk_stride"]
        )
        return {
            "q_ids": q_ids, "q_attn": q_attn, "q_cmask": q_cmask,
            "d_ids": d_ids, "d_attn": d_attn, "d_cmask": d_cmask,
            "qid": s["qid"], "cand_id": s["cand_id"]
        }


def collate_pcr_batch(batch):
    return {
        "q_ids": torch.stack([b["q_ids"] for b in batch]),
        "q_attn": torch.stack([b["q_attn"] for b in batch]),
        "q_cmask": torch.stack([b["q_cmask"] for b in batch]),
        "d_ids": torch.stack([b["d_ids"] for b in batch]),
        "d_attn": torch.stack([b["d_attn"] for b in batch]),
        "d_cmask": torch.stack([b["d_cmask"] for b in batch]),
        "qids": [b["qid"] for b in batch],
        "cand_ids": [b["cand_id"] for b in batch],
    }

train_dataset = LegalPCRDataset(train_queries, train_candidates, tokenizer, CFG)
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    collate_fn=collate_pcr_batch,
    drop_last=(len(train_dataset) >= CFG["batch_size"] and PROFILE != "SMOKE")
)

print(f"✓ Train DataLoader assembled: {len(train_dataset):,} samples, {len(train_loader)} batches/epoch.")


In [ ]:
# 8. Pre-flight Assertion: Verify Distillation Loss & Asymmetric Hardware Flow
print("=== Running Pre-flight Distillation Flow Assertion ===")
teacher_model.eval()
student_model.train()
bridge.train()

assert len(train_loader) > 0, f"DataLoader has 0 batches! Dataset size: {len(train_dataset)}, batch_size: {CFG['batch_size']}"
sample_batch = next(iter(train_loader))

# Micro-slice to at most 2 samples to verify distillation flow safely
with torch.amp.autocast("cuda", dtype=torch.float16):
    # 1. Forward pass on Teacher on teacher_device (cuda:0)
    with torch.no_grad():
        t_q_ids = sample_batch["q_ids"][:1].to(teacher_device)
        t_q_attn = sample_batch["q_attn"][:2].to(teacher_device)
        t_q_cmask = sample_batch["q_cmask"][:2].to(teacher_device)
        t_d_ids = sample_batch["d_ids"][:1].to(teacher_device)
        t_d_attn = sample_batch["d_attn"][:2].to(teacher_device)
        t_d_cmask = sample_batch["d_cmask"][:2].to(teacher_device)
        u_t, v_t = teacher_model(t_q_ids, t_q_attn, t_q_cmask, t_d_ids, t_d_attn, t_d_cmask)

    # Transfer teacher vectors to student_device
    u_t = u_t.to(student_device)
    v_t = v_t.to(student_device)

    # 2. Forward pass on Student & Bridge on student_device (cuda:1)
    s_q_ids = sample_batch["q_ids"][:1].to(student_device)
    s_q_attn = sample_batch["q_attn"][:2].to(student_device)
    s_q_cmask = sample_batch["q_cmask"][:2].to(student_device)
    s_d_ids = sample_batch["d_ids"][:1].to(student_device)
    s_d_attn = sample_batch["d_attn"][:2].to(student_device)
    s_d_cmask = sample_batch["d_cmask"][:2].to(student_device)

    student_model.zero_grad()
    bridge.zero_grad()

    u_s, v_s = student_model(s_q_ids, s_q_attn, s_q_cmask, s_d_ids, s_d_attn, s_d_cmask)
    loss, m_dict = distill_criterion(u_s, v_s, u_t, v_t, bridge)

loss.backward()

assert not torch.isnan(loss), "ERROR: Preflight loss is NaN!"
assert loss.item() > 0.0, f"ERROR: Preflight loss non-positive: {loss.item()}"

# Verify gradients on student model
student_has_grad = any(p.grad is not None and p.grad.abs().sum().item() > 0 for p in student_model.parameters())
assert student_has_grad, "ERROR: Student model received ZERO gradients!"

# Verify gradients on distillation bridge
bridge_has_grad = any(p.grad is not None and p.grad.abs().sum().item() > 0 for p in bridge.parameters())
assert bridge_has_grad, "ERROR: Distillation bridge received ZERO gradients!"

# Verify zero gradients on teacher model
teacher_has_grad = any(p.grad is not None and p.grad.abs().sum().item() > 0 for p in teacher_model.parameters())
assert not teacher_has_grad, "ERROR: Teacher model received gradients (must remain strictly frozen)!"

print(f"✓ Pre-flight Assertion PASSED:")
print(f"    Total Loss:     {m_dict['loss_total']:.4f}")
print(f"    InfoNCE Loss:   {m_dict['loss_infonce']:.4f}")
print(f"    KL Div (T=2.0): {m_dict['loss_kl']:.4f}")
print(f"    Cosine Loss:    {m_dict['loss_cos']:.4f}")
print("✓ Teacher is strictly frozen; Student & Bridge are actively learning.")

student_model.zero_grad()
bridge.zero_grad()


In [ ]:
# 9. Continuous Cloud Sync Callback & Automatic Crash Recovery Engine
from huggingface_hub import list_repo_files

class HfContinuousSyncCallback:
    """
    Threaded, non-blocking cloud synchronization of checkpoints to Hugging Face Hub.
    Ensures that every checkpoint-{step} is persisted without delaying the training loop.
    """
    def __init__(self, repo_id: str, subfolder: str, token: Optional[str] = None):
        self.repo_id = repo_id
        self.subfolder = subfolder
        self.token = token
        self.api = HfApi()

    def sync_checkpoint_async(self, checkpoint_dir: str, step: int):
        if not self.token:
            print("⚠ HF_TOKEN not set; skipping remote sync.")
            return

        def _upload_task():
            try:
                ckpt_folder_name = os.path.basename(checkpoint_dir)
                remote_path = f"{self.subfolder}/{ckpt_folder_name}"
                print(f"[CloudSync] Starting upload for step {step} -> {self.repo_id}/{remote_path}...")
                self.api.upload_folder(
                    folder_path=checkpoint_dir,
                    path_in_repo=remote_path,
                    repo_id=self.repo_id,
                    token=self.token
                )
                print(f"[CloudSync] ✓ Step {step} successfully pushed to Hugging Face.")
            except Exception as e:
                print(f"[CloudSync] ⚠ Warning: Cloud sync failed for step {step}: {e}")

        thread = threading.Thread(target=_upload_task, daemon=True)
        thread.start()


sync_callback = HfContinuousSyncCallback(
    repo_id=HF_MASTER_REPO,
    subfolder=HUB_SUBFOLDER,
    token=HF_TOKEN
)


def find_latest_checkpoint(local_ckpt_dir: str, hf_repo: str, hf_subfolder: str) -> Tuple[Optional[str], int]:
    """
    Finds the latest available checkpoint locally, or checks remote HF Hub repo if local is empty.
    """
    local_ckpts = glob.glob(os.path.join(local_ckpt_dir, "checkpoint-*"))
    max_step = -1
    best_ckpt = None

    for c in local_ckpts:
        try:
            step = int(os.path.basename(c).replace("checkpoint-", ""))
            if step > max_step:
                max_step = step
                best_ckpt = c
        except Exception:
            pass

    if best_ckpt:
        print(f"✓ Found local checkpoint: {best_ckpt} (Step {max_step})")
        return best_ckpt, max_step

    # Check remote Hugging Face Hub
    if HF_TOKEN:
        try:
            print(f"Querying remote repo {hf_repo} under {hf_subfolder} for checkpoints...")
            files = list_repo_files(repo_id=hf_repo, token=HF_TOKEN)
            remote_steps = []
            for f in files:
                if f.startswith(hf_subfolder) and "checkpoint-" in f:
                    parts = f.split("/")
                    for p in parts:
                        if p.startswith("checkpoint-"):
                            try:
                                s = int(p.replace("checkpoint-", ""))
                                remote_steps.append((s, p))
                            except Exception:
                                pass
            if remote_steps:
                remote_steps.sort(key=lambda x: x[0], reverse=True)
                highest_step, folder_name = remote_steps[0]
                print(f"Found remote checkpoint at Step {highest_step}. Downloading...")
                dl_dir = os.path.join(local_ckpt_dir, folder_name)
                os.makedirs(dl_dir, exist_ok=True)
                snapshot_download(
                    repo_id=hf_repo,
                    allow_patterns=[f"{hf_subfolder}/{folder_name}/*"],
                    local_dir=local_ckpt_dir,
                    token=HF_TOKEN
                )
                nested = os.path.join(local_ckpt_dir, hf_subfolder, folder_name)
                if os.path.exists(nested):
                    best_ckpt = nested
                else:
                    best_ckpt = dl_dir
                return best_ckpt, highest_step
        except Exception as e:
            print(f"Remote checkpoint check failed: {e}")

    return None, 0


In [ ]:
# 10. Multi-Task Knowledge Distillation Training Loop
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

trainable_params = list(student_model.parameters()) + list(bridge.parameters())
optimizer = AdamW(trainable_params, lr=4e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

total_steps = CFG["train_epochs"] * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=max(total_steps, 1), eta_min=1e-6)

# Check for existing checkpoint to resume
latest_ckpt, resume_step = find_latest_checkpoint(CHECKPOINT_DIR, HF_MASTER_REPO, HUB_SUBFOLDER)
start_step = 0
best_f1 = 0.0

if latest_ckpt and os.path.exists(os.path.join(latest_ckpt, "student_model.pt")):
    try:
        print(f"Resuming distillation training from checkpoint: {latest_ckpt} at step {resume_step}...")
        student_model.load_state_dict(torch.load(os.path.join(latest_ckpt, "student_model.pt"), map_location=student_device))
        bridge_path = os.path.join(latest_ckpt, "bridge.pt")
        if os.path.exists(bridge_path):
            bridge.load_state_dict(torch.load(bridge_path, map_location=student_device))
        opt_path = os.path.join(latest_ckpt, "optimizer.pt")
        if os.path.exists(opt_path):
            optimizer.load_state_dict(torch.load(opt_path, map_location=student_device))
        sched_path = os.path.join(latest_ckpt, "scheduler.pt")
        if os.path.exists(sched_path):
            scheduler.load_state_dict(torch.load(sched_path, map_location=student_device))
        state_info_path = os.path.join(latest_ckpt, "state.json")
        if os.path.exists(state_info_path):
            with open(state_info_path, "r") as sf:
                s_dict = json.load(sf)
                best_f1 = s_dict.get("best_f1", 0.0)
        start_step = resume_step
        print(f"✓ Resumed successfully from step {start_step}. Previous Best F1: {best_f1:.4f}")
    except Exception as e:
        print(f"⚠ Could not restore checkpoint state cleanly: {e}. Starting from step 0.")
        start_step = 0


def evaluate_student(eval_model, queries: List[dict], candidates: Dict[str, str], tokenizer, cfg: dict) -> Tuple[float, float, float]:
    """
    Evaluates Micro-F1@7, Micro-Precision@7, and Micro-Recall@7 on the dev set using student model.
    """
    eval_model.eval()
    cand_ids = list(candidates.keys())
    if not cand_ids or not queries:
        return 0.0, 0.0, 0.0

    # 1. Pre-encode candidate pool
    cand_embs = []
    with torch.no_grad():
        c_batch_size = 32
        for i in range(0, len(cand_ids), c_batch_size):
            batch_cids = cand_ids[i:i + c_batch_size]
            b_ids, b_attn, b_cmask = [], [], []
            for cid in batch_cids:
                c_ids, c_attn, c_cmask = chunk_document(
                    candidates[cid], tokenizer,
                    max_chunks=cfg["max_chunks"],
                    chunk_size=cfg["chunk_size"],
                    chunk_stride=cfg["chunk_stride"]
                )
                b_ids.append(c_ids)
                b_attn.append(c_attn)
                b_cmask.append(c_cmask)

            c_ids_t = torch.stack(b_ids).to(student_device)
            c_attn_t = torch.stack(b_attn).to(student_device)
            c_cmask_t = torch.stack(b_cmask).to(student_device)

            e = eval_model.encode_chunks(c_ids_t, c_attn_t, c_cmask_t)
            cand_embs.append(e.cpu())

    cand_mat = torch.cat(cand_embs, dim=0)  # [Num_Cands, D]

    # 2. Encode queries and compute Top 7 predictions
    total_tp = 0
    total_pred = 0
    total_gold = 0

    with torch.no_grad():
        for q in queries:
            gold_set = set(q.get("gold_cids", []))
            if not gold_set:
                continue

            q_ids, q_attn, q_cmask = chunk_document(
                q["text"], tokenizer,
                max_chunks=cfg["max_chunks"],
                chunk_size=cfg["chunk_size"],
                chunk_stride=cfg["chunk_stride"]
            )
            q_ids_t = q_ids.unsqueeze(0).to(student_device)
            q_attn_t = q_attn.unsqueeze(0).to(student_device)
            q_cmask_t = q_cmask.unsqueeze(0).to(student_device)

            q_emb = eval_model.encode_chunks(q_ids_t, q_attn_t, q_cmask_t)

            sims = torch.matmul(q_emb.cpu(), cand_mat.T).squeeze(0)  # [Num_Cands]
            topk_idx = torch.topk(sims, k=min(7, len(cand_ids))).indices.tolist()
            pred_cids = [cand_ids[idx] for idx in topk_idx]

            tp = len(set(pred_cids).intersection(gold_set))
            total_tp += tp
            total_pred += len(pred_cids)
            total_gold += len(gold_set)

    prec = total_tp / max(total_pred, 1)
    rec = total_tp / max(total_gold, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    return f1, prec, rec


# Main Distillation Loop
train_history = []
global_step = start_step
eval_interval = CFG["eval_steps"]
save_interval = CFG["save_steps"]

print(f"=== Starting Distillation Training Loop (Target Steps: {total_steps}) ===")
epoch = 0

while global_step < total_steps:
    epoch += 1
    student_model.train()
    bridge.train()
    teacher_model.eval()

    for step_idx, batch in enumerate(train_loader):
        global_step += 1
        if global_step <= start_step:
            continue

        with torch.amp.autocast("cuda", dtype=torch.float16):
            # 1. Forward teacher on teacher_device (cuda:0)
            with torch.no_grad():
                t_q_ids = batch["q_ids"].to(teacher_device)
                t_q_attn = batch["q_attn"].to(teacher_device)
                t_q_cmask = batch["q_cmask"].to(teacher_device)
                t_d_ids = batch["d_ids"].to(teacher_device)
                t_d_attn = batch["d_attn"].to(teacher_device)
                t_d_cmask = batch["d_cmask"].to(teacher_device)
                u_t, v_t = teacher_model(t_q_ids, t_q_attn, t_q_cmask, t_d_ids, t_d_attn, t_d_cmask)

            # Transfer teacher representations to student_device (cuda:1)
            u_t = u_t.to(student_device)
            v_t = v_t.to(student_device)

            # 2. Forward student on student_device (cuda:1)
            s_q_ids = batch["q_ids"].to(student_device)
            s_q_attn = batch["q_attn"].to(student_device)
            s_q_cmask = batch["q_cmask"].to(student_device)
            s_d_ids = batch["d_ids"].to(student_device)
            s_d_attn = batch["d_attn"].to(student_device)
            s_d_cmask = batch["d_cmask"].to(student_device)

            optimizer.zero_grad()
            u_s, v_s = student_model(s_q_ids, s_q_attn, s_q_cmask, s_d_ids, s_d_attn, s_d_cmask)
            loss, m_dict = distill_criterion(u_s, v_s, u_t, v_t, bridge)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        if global_step % 20 == 0 or global_step == 1:
            curr_lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch [{epoch}] | Step [{global_step}/{total_steps}] | Total Loss: {m_dict['loss_total']:.4f} "
                  f"(InfoNCE: {m_dict['loss_infonce']:.3f}, KL: {m_dict['loss_kl']:.3f}, Cos: {m_dict['loss_cos']:.3f}) | LR: {curr_lr:.2e}")

        # Evaluation & Checkpointing
        if global_step % eval_interval == 0 or global_step == total_steps:
            val_f1, val_prec, val_rec = evaluate_student(student_model, dev_queries, dev_candidates, tokenizer, CFG)
            print(f"--- [Validation @ Step {global_step}] ---")
            print(f"    Student Micro-F1@7:        {val_f1:.4f}")
            print(f"    Student Micro-Precision@7: {val_prec:.4f}")
            print(f"    Student Micro-Recall@7:    {val_rec:.4f}")

            train_history.append({
                "step": global_step,
                "loss_total": m_dict["loss_total"],
                "loss_infonce": m_dict["loss_infonce"],
                "loss_kl": m_dict["loss_kl"],
                "loss_cos": m_dict["loss_cos"],
                "val_f1": val_f1,
                "val_prec": val_prec,
                "val_rec": val_rec
            })

            if val_f1 > best_f1:
                best_f1 = val_f1
                print(f"    ⭐ New Best Dev Micro-F1@7: {best_f1:.4f}! Saving student checkpoint...")

            # Save checkpoint
            if global_step % save_interval == 0 or global_step == total_steps:
                ckpt_dir = os.path.join(CHECKPOINT_DIR, f"checkpoint-{global_step}")
                os.makedirs(ckpt_dir, exist_ok=True)
                
                torch.save(student_model.state_dict(), os.path.join(ckpt_dir, "student_model.pt"))
                torch.save(bridge.state_dict(), os.path.join(ckpt_dir, "bridge.pt"))
                torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))
                torch.save(scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.pt"))
                
                state_info = {
                    "step": global_step,
                    "epoch": epoch,
                    "val_f1": val_f1,
                    "best_f1": best_f1,
                    "loss_metrics": m_dict
                }
                with open(os.path.join(ckpt_dir, "state.json"), "w") as sf:
                    json.dump(state_info, sf, indent=2)

                print(f"✓ Saved local checkpoint: {ckpt_dir}")
                # Non-blocking continuous upload to Hugging Face
                sync_callback.sync_checkpoint_async(ckpt_dir, global_step)

            student_model.train()
            bridge.train()

        if global_step >= total_steps:
            break

print("✓ Knowledge Distillation completed successfully.")


In [ ]:
# 11. Plot Distillation Loss Components & Student Evaluation Curves
import matplotlib.pyplot as plt

if train_history:
    steps = [h["step"] for h in train_history]
    total_losses = [h["loss_total"] for h in train_history]
    infonce_losses = [h["loss_infonce"] for h in train_history]
    kl_losses = [h["loss_kl"] for h in train_history]
    cos_losses = [h["loss_cos"] for h in train_history]
    f1s = [h["val_f1"] for h in train_history]
    precs = [h["val_prec"] for h in train_history]
    recs = [h["val_rec"] for h in train_history]

    plt.figure(figsize=(15, 5))

    # Loss decomposition
    plt.subplot(1, 2, 1)
    plt.plot(steps, total_losses, marker='o', color='purple', label="Total Multi-Task Loss")
    plt.plot(steps, infonce_losses, marker='.', linestyle='--', color='blue', label="0.5 * InfoNCE")
    plt.plot(steps, kl_losses, marker='.', linestyle='--', color='red', label="0.4 * T^2 * KL")
    plt.plot(steps, cos_losses, marker='.', linestyle='--', color='cyan', label="0.1 * Cosine Align")
    plt.title("Student Multi-Task Distillation Loss Decomposition")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

    # Metrics
    plt.subplot(1, 2, 2)
    plt.plot(steps, f1s, marker='s', color='green', label="Student Dev Micro-F1@7")
    plt.plot(steps, precs, marker='^', color='orange', label="Student Dev Micro-Precision@7")
    plt.plot(steps, recs, marker='v', color='darkblue', label="Student Dev Micro-Recall@7")
    plt.title("Student Retrieval Evaluation @ 7")
    plt.xlabel("Step")
    plt.ylabel("Score")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()

    plt.tight_layout()
    plt.savefig(LOSS_PLOT_PATH, dpi=300)
    plt.show()
    print(f"✓ Saved training curves to: {LOSS_PLOT_PATH}")

    with open(METRICS_JSON_PATH, "w") as f:
        json.dump(train_history, f, indent=2)
else:
    print("No training steps executed in this run (or already finished).")


In [ ]:
# 12. Student Model Test Candidate Pre-Encoding & Top 7 Candidate Ranking
print("=== Generating Test Predictions using Compact Distilled Student (110M) ===")
student_model.eval()

test_cids = list(test_candidates.keys())
print(f"Encoding {len(test_cids):,} Test Candidates with Student Model on {student_device}...")

cand_vectors = []
c_batch_size = 32

with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    for start_idx in range(0, len(test_cids), c_batch_size):
        batch_keys = test_cids[start_idx:start_idx + c_batch_size]
        b_ids, b_attn, b_cmask = [], [], []
        for ck in batch_keys:
            c_ids, c_attn, c_cmask = chunk_document(
                test_candidates[ck], tokenizer,
                max_chunks=CFG["max_chunks"],
                chunk_size=CFG["chunk_size"],
                chunk_stride=CFG["chunk_stride"]
            )
            b_ids.append(c_ids)
            b_attn.append(c_attn)
            b_cmask.append(c_cmask)

        t_ids = torch.stack(b_ids).to(student_device)
        t_attn = torch.stack(b_attn).to(student_device)
        t_cmask = torch.stack(b_cmask).to(student_device)

        embs = student_model.encode_chunks(t_ids, t_attn, t_cmask)
        cand_vectors.append(embs.cpu())

cand_matrix = torch.cat(cand_vectors, dim=0)  # [Total_Candidates, D_student]
print(f"✓ Encoded Student Candidate Matrix: {cand_matrix.shape}")

# Encode test queries and compute Top 7 candidate IDs per query
predictions = {}
print(f"Ranking Top 7 Candidates for {len(test_queries):,} Test Queries...")

with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    for q in test_queries:
        qid = q["qid"]
        q_ids, q_attn, q_cmask = chunk_document(
            q["text"], tokenizer,
            max_chunks=CFG["max_chunks"],
            chunk_size=CFG["chunk_size"],
            chunk_stride=CFG["chunk_stride"]
        )
        q_ids_t = q_ids.unsqueeze(0).to(student_device)
        q_attn_t = q_attn.unsqueeze(0).to(student_device)
        q_cmask_t = q_cmask.unsqueeze(0).to(student_device)

        q_emb = student_model.encode_chunks(q_ids_t, q_attn_t, q_cmask_t)

        sim_scores = torch.matmul(q_emb.cpu(), cand_matrix.T).squeeze(0)  # [Total_Candidates]
        top7_indices = torch.topk(sim_scores, k=min(7, len(test_cids))).indices.tolist()
        top7_cids = [test_cids[i] for i in top7_indices]

        # In case test pool has fewer than 7 candidates
        while len(top7_cids) < 7 and test_cids:
            for fallback_cid in test_cids:
                if fallback_cid not in top7_cids:
                    top7_cids.append(fallback_cid)
                if len(top7_cids) == 7:
                    break

        predictions[qid] = top7_cids[:7]

print(f"✓ Generated predictions for {len(predictions):,} queries using Student Model.")


In [ ]:
# 13. Strict CodaBench Output Verification, Packaging & Master Hub Sync
print("=== Strict CodaBench Verification ===")

# Save prediction.json
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)
print(f"✓ Saved prediction.json to: {PREDICTION_PATH}")

# Verification 1: File exists and is non-empty
assert os.path.exists(PREDICTION_PATH), "ERROR: prediction.json not found!"
with open(PREDICTION_PATH, "r", encoding="utf-8") as f:
    loaded_preds = json.load(f)

# Verification 2: Every query has exactly 7 unique candidate IDs
test_cids_set = set(test_candidates.keys())
for qid, cids in loaded_preds.items():
    assert len(cids) == 7, f"ERROR: Query {qid} does not have exactly 7 candidates (got {len(cids)})!"
    assert len(set(cids)) == 7, f"ERROR: Query {qid} contains duplicate candidate IDs!"
    for cid in cids:
        assert cid in test_cids_set, f"ERROR: Candidate {cid} for query {qid} is not in test_candidates.json!"

print("✓ CodaBench Assertions PASSED:")
print(f"  - Total Queries Predicted: {len(loaded_preds):,}")
print("  - Exactly 7 unique candidates per query: VERIFIED.")
print("  - All candidates belong to official test_candidates.json: VERIFIED.")

# Package submission.zip with prediction.json directly at root
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"✓ Successfully packaged submission.zip: {SUBMISSION_ZIP}")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as check_zip:
    namelist = check_zip.namelist()
    assert namelist == ["prediction.json"], f"ERROR: submission.zip root structure invalid: {namelist}"
print("✓ Root packaging verified: 'prediction.json' is at the root level.")

# Save final distilled student model weights
final_student_dir = os.path.join(OUTPUT_DIR, "final_distilled_student")
os.makedirs(final_student_dir, exist_ok=True)
torch.save(student_model.state_dict(), os.path.join(final_student_dir, "student_biencoder.pt"))
torch.save(bridge.state_dict(), os.path.join(final_student_dir, "bridge.pt"))
print(f"✓ Final distilled student model saved to: {final_student_dir}")

# Push final artifacts to Hugging Face Hub under Phase_7/sub06_model4_to_model3_distill/
if HF_TOKEN:
    try:
        api = HfApi()
        print(f"Uploading final Phase 7 Sub-06 artifacts to {HF_MASTER_REPO}/{HUB_SUBFOLDER}...")
        api.upload_file(
            path_or_fileobj=PREDICTION_PATH,
            path_in_repo=f"{HUB_SUBFOLDER}/prediction.json",
            repo_id=HF_MASTER_REPO,
            token=HF_TOKEN
        )
        api.upload_file(
            path_or_fileobj=SUBMISSION_ZIP,
            path_in_repo=f"{HUB_SUBFOLDER}/submission.zip",
            repo_id=HF_MASTER_REPO,
            token=HF_TOKEN
        )
        if os.path.exists(LOSS_PLOT_PATH):
            api.upload_file(
                path_or_fileobj=LOSS_PLOT_PATH,
                path_in_repo=f"{HUB_SUBFOLDER}/loss_and_metrics_plot.png",
                repo_id=HF_MASTER_REPO,
                token=HF_TOKEN
            )
        api.upload_folder(
            folder_path=final_student_dir,
            path_in_repo=f"{HUB_SUBFOLDER}/final_model",
            repo_id=HF_MASTER_REPO,
            token=HF_TOKEN
        )
        print("✓ All final Phase 7 Sub-06 artifacts uploaded to Hugging Face Master Repository.")
    except Exception as e:
        print(f"⚠ Master Hub upload failed: {e}")
else:
    print("⚠ HF_TOKEN not present. Skipping final master upload.")

print("=== CS781 Phase 7 Sub-06 Execution Complete ===")
